In [1]:
import sys
sys.path.append("../")

In [2]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload
from qmpsqsc.models.data.utils import flip_sites_in_mps
import qmpsqsc.models.data as mpsdata

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [3]:
import torch.nn.functional as F

L = 30
chi = 2
d = 2
ghz = mpsqsc.build_ghz_state(L, d, chi).to(dtype=torch.complex128)
ghz = ghz.normalize()

zghz = ghz.copy()
zghz.As[0][:, 1] = -ghz.As[0][:, 1]

ghz_X_errors = [flip_sites_in_mps(ghz, [i], "X") for i in range(L)]
ghz_Y_errors = [flip_sites_in_mps(ghz, [i], "Y") for i in range(L)]
ghz_Z_errors = [flip_sites_in_mps(ghz, [i], "Z") for i in range(L)]

zghz_X_errors = [flip_sites_in_mps(zghz, [i], "X") for i in range(L)]
zghz_Y_errors = [flip_sites_in_mps(zghz, [i], "Y") for i in range(L)]
zghz_Z_errors = [flip_sites_in_mps(zghz, [i], "Z") for i in range(L)]

allup = mpsqsc.build_classical_state(L, d, [0]*L).to(dtype=torch.complex128)
alldown = mpsqsc.build_classical_state(L, d, [1]*L).to(dtype=torch.complex128)


In [4]:
ghzs_X = mpsqsc.add_mpstates(ghz_X_errors)
ghzs_Y = mpsqsc.add_mpstates(ghz_Y_errors)



zghzs_X = mpsqsc.add_mpstates(zghz_X_errors)
zghzs_Y = mpsqsc.add_mpstates(zghz_Y_errors)


In [5]:
from qmpsqsc.models.mpsqsc.compress import compress_mpstate

ghzs_X, ghzsX_fid = compress_mpstate(ghzs_X, 4, adam_steps=0, n_sweeps=1)
ghzs_Y, ghzsY_fid = compress_mpstate(ghzs_Y, 4, adam_steps=0, n_sweeps=1)

print("fidelities", ghzsX_fid, ghzsY_fid)

zghzs_X, zghzX_fid = compress_mpstate(zghzs_X, 4, adam_steps=0, n_sweeps=1)
zghzs_Y, zghzY_fid = compress_mpstate(zghzs_Y, 4, adam_steps=0, n_sweeps=1)

print("fidelities", zghzX_fid, zghzY_fid)


ghzs = mpsqsc.add_mpstates([ghz, ghzs_X])
ghzs = ghzs.normalize()

ghzs, ghzs_fid = compress_mpstate(ghzs, 6, adam_steps=0, n_sweeps=1)

print("fidelities", ghzs_fid)

zghzs = mpsqsc.add_mpstates([zghz, zghzs_X])
zghzs = zghzs.normalize()

zghzs, zghz_fid = compress_mpstate(zghzs, 6, adam_steps=0, n_sweeps=1)

print("fidelities", zghz_fid)


/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)
/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/algorithms/mps_common.py:2259: UserWarning: VariationalCompression with min_sweeps=max_sweeps: we recommend to set tol_theta_diff=None to avoid overhead
  warnings.warn(
/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/compress.py:36: ComplexWarning: Casting complex values to real discards the imaginary part
  return res_mps, float(psi_t.overlap(psi))


fidelities (1.0000000000000022+0j) (1.0000000000000042-1.5299722524569638e-16j)
fidelities (1.000000000000004+0j) (1.000000000000003-7.965995054199865e-16j)
fidelities (1.0000000000000004+0j)
fidelities (1.0000000000000013+0j)


In [6]:
data_generator = mpsdata.ghz.create_ghz_rho_batch_qsc(ghz, allup, alldown, 2**6, 0.5, random_flip=True)


In [7]:
ghzs.overlap(ghz_X_errors[0]).abs(), zghzs.overlap(ghz_X_errors[0]).abs()

(tensor(0.1291, dtype=torch.float64), tensor(8.3267e-17, dtype=torch.float64))

In [8]:
ghzs.overlap(ghz).abs(), zghzs.overlap(ghz).abs()

(tensor(0.7071, dtype=torch.float64), tensor(1.3323e-15, dtype=torch.float64))

In [9]:
from torch import nn
import torch

# Classifier head: V (2D) -> probabilities over {0, 1}
class ClassifierHead(nn.Module):
    def __init__(self, in_dim: int = 2, hidden_dim: int = 16, dtype=torch.float64, device=torch.device("cpu")):
        super().__init__()
        # If dtype or device are specified, pass them to nn.Linear
        linear_args = {}
        if dtype is not None:
            linear_args['dtype'] = dtype
        if device is not None:
            linear_args['device'] = device

        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, **linear_args),
            nn.ReLU(),
            nn.Linear(hidden_dim, 4, **linear_args),  # logits for 2 classes
            nn.ReLU(),
            nn.Linear(4, 2, **linear_args),
        )

    def forward(self, V):
        logits = self.net(V)               # shape (..., 2)
        return logits                      # during training, return logits

    def predict_proba(self, V):
        logits = self.net(V)
        probs = torch.nn.functional.softmax(logits, dim=-1)  # convert to probabilities
        return probs

In [10]:
ch = ClassifierHead()
criterion = nn.CrossEntropyLoss()

In [11]:
ghzs.set_requires_grad(True)
zghzs.set_requires_grad(True)
optimizer = torch.optim.Adam(ghzs.As + zghzs.As, lr=0.001)
optim_nn = torch.optim.Adam(ch.parameters(), lr=0.01)

optimizer.zero_grad()
optim_nn.zero_grad()

for _ in range(1000):
    states, labels, _ = next(data_generator)
    # loss, acc = mpsdata.calculate_loss_mpstates(ghzs, zghzs, states, labels)
    probs, norms = mpsdata.mps_binary_predict(ghzs, zghzs, states)
    ll_nn = ch(probs)
    loss = criterion(ll_nn, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    optim_nn.step()
    optim_nn.zero_grad()
    acc = (ll_nn.argmax(dim=-1) == labels).float()
    print(loss.item(), acc.mean().item())


0.6866901359179484 0.5
0.6835422998699253 0.5
0.6792450339227311 0.5
0.6745756072581547 0.859375
0.6684248326541643 0.890625
0.6678557124000815 1.0
0.6666746856120744 1.0
0.655830701105562 0.8125
0.645859616888991 0.84375
0.6278908539758976 0.90625
0.6258473975299785 0.859375
0.6374528651577627 0.765625
0.6145265988145239 0.828125
0.6054613139127085 0.828125
0.6232351567398953 0.75
0.5602875154162069 0.890625
0.5681500473886284 0.84375
0.5894247430432018 0.78125
0.5025655381351528 0.921875
0.5250465264426787 0.859375
0.4919490216599572 0.890625
0.5507018336016659 0.796875
0.5289466372094667 0.8125
0.46427141734154315 0.875
0.4372202927355093 0.890625
0.4745915166248213 0.84375
0.3960335779446049 0.90625
0.45794209543549386 0.84375
0.45036892390144084 0.84375
0.48300871455113126 0.8125
0.4743968428167844 0.8125
0.4050565916917227 0.859375
0.4987400416978393 0.78125
0.35581144134955656 0.890625
0.3700170411900825 0.875
0.47596445576283875 0.78125
0.4711183154371562 0.78125
0.467094561143

KeyboardInterrupt: 

In [31]:
reload(mpsqsc.helper)

<module 'qmpsqsc.models.mpsqsc.helper' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/helper.py'>

In [32]:
mps_qsc = mpsqsc.helper.build_qsc_from_mpstates(ghzs, zghzs)
mps_qsc = mps_qsc.truncate_bond_dimension(2)
mps_qsc.set_requires_grad(True)

In [36]:
predict, norm = mps_qsc.predict([ghz_Z_errors[2]])

ch(predict)

tensor([[ 0.3187, -0.7837]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

In [37]:
optimizer = torch.optim.Adam(mps_qsc.As, lr=0.001)
optim_nn = torch.optim.Adam(ch.parameters(), lr=0.01)

optimizer.zero_grad()
optim_nn.zero_grad()

for _ in range(1000):
    states, labels, _ = next(data_generator)
    probs, norms = mps_qsc.predict(states)
    ll_nn = ch(probs)
    loss = criterion(ll_nn, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    optim_nn.step()
    optim_nn.zero_grad()
    acc = (ll_nn.argmax(dim=-1) == labels).float()
    print(loss.item(), acc.mean().item())


0.05243714398043173 1.0
0.024429291096466178 1.0
0.02077939969964316 1.0
0.019242269820096816 1.0
0.016856831809282786 1.0
0.016748647929127888 1.0
0.013240260250161175 1.0
0.01040661154361591 1.0
0.009664738956864953 1.0
0.008078682700643785 1.0
0.00754854175780423 1.0
0.007865605892377111 1.0
0.007237955358576404 1.0
0.006479264872086649 1.0
0.007015681891480054 1.0
0.0062960809299053115 1.0
0.004783716586510625 1.0
0.005259386689204217 1.0
0.004184532108307984 1.0
0.0036819642890586696 1.0
0.004090807779863453 1.0
0.003438700154603423 1.0
0.005144918430615774 1.0
0.006084761462634122 1.0
0.0023157458672822878 1.0
0.00342714608903677 1.0
0.00263683213097539 1.0
0.00227969374692318 1.0
0.002415317527878342 1.0
0.002085116419257899 1.0
0.0017548423573996825 1.0
0.0022125393181672776 1.0
0.002027365808398065 1.0
0.0019448033104838395 1.0
0.0017712696640688433 1.0
0.0018276271587606165 1.0
0.0018015048178378777 1.0
0.0017481221861877893 1.0
0.0016247332002240261 1.0
0.0014536166094270653

KeyboardInterrupt: 

In [55]:
mps_qsc = mps_qsc.canonicalize(truncate=True)
Us, last = qmps.construct_unitary_from_As(mps_qsc.As)
qmps_ghz = qmps.qMPS(L, chi, d, Us=Us, last_unitary=last)

In [84]:
w = 1
qmps_ghz.set_weights(w)

probs, norms = qmps_ghz.predict(states)
ll_nn = ch(probs)
loss = criterion(ll_nn, labels)
print(loss.item())

0.03471545684131284


In [86]:
labels

tensor([0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1,
        1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0,
        0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0])

In [83]:
from qmpsqsc.models.qmps.optimizer import StiefelAdam

optimizer = StiefelAdam(qmps_ghz.unitaries(), lr=0.001)
optimizer_nn = torch.optim.Adam(ch.parameters(), lr=0.001)

for step in range(1000):
    states, labels, _ = next(data_generator)
    optimizer.zero_grad(set_to_none=True)
    optimizer_nn.zero_grad(set_to_none=True)

    # Shuffle the 4 examples each step
    probs, norms = qmps_ghz.predict(states)
    ll_nn = ch(probs)
    loss = criterion(ll_nn, labels)
    loss.backward()
    optimizer.step()
    optimizer_nn.step()

    acc = (ll_nn.argmax(dim=-1) == labels).float()

    print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc.mean().item():.3f}")


[step     0] loss=0.303773  acc=0.938
[step     1] loss=0.210825  acc=0.953
[step     2] loss=0.061583  acc=0.984
[step     3] loss=0.240504  acc=0.938
[step     4] loss=0.100686  acc=0.953
[step     5] loss=0.147619  acc=0.969
[step     6] loss=0.282536  acc=0.953
[step     7] loss=0.140315  acc=0.969
[step     8] loss=0.168926  acc=0.953
[step     9] loss=0.107894  acc=0.953
[step    10] loss=0.113046  acc=0.969
[step    11] loss=0.184203  acc=0.953
[step    12] loss=0.222017  acc=0.969
[step    13] loss=0.310271  acc=0.938
[step    14] loss=0.151861  acc=0.969
[step    15] loss=0.041729  acc=1.000
[step    16] loss=0.355616  acc=0.938
[step    17] loss=0.200205  acc=0.953
[step    18] loss=0.230814  acc=0.922
[step    19] loss=0.198545  acc=0.953
[step    20] loss=0.189471  acc=0.953
[step    21] loss=0.065759  acc=0.984
[step    22] loss=0.062859  acc=0.984
[step    23] loss=0.174622  acc=0.969
[step    24] loss=0.163513  acc=0.953
[step    25] loss=0.214031  acc=0.938
[step    26]